# Module 3 — Appendiceal Cancer: Exploration

This module takes an **indication-first** approach, in contrast to Module 1 which was drug-first.
Rather than starting with a specific drug and asking what happened to those patients,
this starts with a specific cancer population - appendiceal cancer. It ask what drugs
were reported and what adverse events occurred.

**Why appendiceal cancer?**
Appendiceal cancer is a rare malignancy often treated with cytoreductive surgery (CRS)
combined with HIPEC (hyperthermic intraperitoneal chemotherapy). FAERS reports for this
population are sparse relative to common cancers, which itself is a clinically meaningful
finding about the limits of spontaneous reporting for rare diseases.

**Analyses in this module:**
- Indication term discovery and appendiceal-specific filtering
- Build `appendiceal_reports` - the filtered primaryid set used across all notebooks
- Drug landscape: what drugs are most reported in this population?
- Data sparsity assessment

**Prerequisite:** `faers.db` must be present at the path defined in the setup cell below.

In [2]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

db_path = r"C:\Users\palla\OneDrive\Documents\Coding Projects\FDA_FAERS\database\faers.db"
conn = sqlite3.connect(db_path)

In [3]:
#SQL query to extract data for appendiceal cancer indication


appendiceal_cancer_indi = pd.read_sql_query("""
SELECT indi_pt, COUNT(*) AS count
                                            FROM indi
                                            WHERE indi_pt LIKE '%appendic%' OR indi_pt LIKE '%appendiceal%' OR indi_pt LIKE '%appendix%' OR indi_pt LIKE '%Mucin%' OR indi_pt LIKE 'Append'
                                            GROUP BY indi_pt
                                            ORDER BY count DESC;
""", conn)

appendiceal_cancer_indi.head(20)


,indi_pt,count
0,Adenocarcinoma of appendix,199
1,Appendicitis,100
2,Appendix cancer,97
3,Appendicectomy,41
4,Mucinous breast carcinoma,38
5,Appendicitis perforated,38
6,Appendix cancer metastatic,21
7,Mucinous adenocarcinoma of appendix,14
8,Follicular mucinosis,14
9,Mucinous cystadenocarcinoma ovary,9


## Step 1: Indication Term Discovery

Scan the `indi` table for all terms matching appendiceal-related keywords. We review the full output before committing to a final term list — this prevents accidentally including non-cancer conditions (appendicitis, carcinoids at non-appendiceal sites, etc.).

In [4]:
# Keyword scan of the indi table to surface all appendiceal-related indication terms
# LIKE wildcards on both sides (%term%) match the keyword anywhere in the string
# We cast indi_pt to LOWER to make the search case-insensitive
appendiceal_indi_terms = pd.read_sql_query("""
    SELECT indi_pt, COUNT(*) AS reports
    FROM indi
    WHERE indi_pt LIKE '%appendic%'
       OR indi_pt LIKE '%mucin%'
       OR indi_pt LIKE '%pseudomyxoma%'
       OR indi_pt LIKE '%goblet cell%'
       OR indi_pt LIKE '%carcinoid%'
    GROUP BY indi_pt
    ORDER BY reports DESC
""", conn)

appendiceal_indi_terms

,indi_pt,reports
0,Carcinoid tumour,692
1,Carcinoid syndrome,306
2,Appendicitis,100
3,Carcinoid tumour pulmonary,77
4,Carcinoid tumour of the small bowel,71
5,Metastatic carcinoid tumour,44
6,Appendicectomy,41
7,Mucinous breast carcinoma,38
8,Appendicitis perforated,38
9,Carcinoid tumour of the gastrointestinal tract,34


## Step 2: Manual Curation

From the discovery output above, we keep only confirmed appendiceal cancer terms and exclude:
- **Carcinoids without site qualifier** (`Carcinoid tumour`, `Carcinoid syndrome`) — carcinoids arise throughout the GI tract, lung, and stomach; too broad without a site
- **Explicitly non-appendiceal carcinoids** (`pulmonary`, `small bowel`, `stomach`, `liver`)
- **Non-cancer appendiceal terms** (`Appendicitis`, `Appendicitis perforated`, `Appendicectomy`)
- **Non-appendiceal mucinous tumors** (`Mucinous breast carcinoma`, `Mucinous cystadenocarcinoma ovary`, `Follicular mucinosis`)

**Retained terms (~339 reports total):**

| Term | Reports |
|---|---|
| Adenocarcinoma of appendix | 199 |
| Appendix cancer | 97 |
| Appendix cancer metastatic | 21 |
| Mucinous adenocarcinoma of appendix | 14 |
| Pseudomyxoma peritonei | 8 |

> **Note on data sparsity:** 339 total reports reflects the real-world rarity of appendiceal cancer. FAERS spontaneous reporting is known to under-represent rare malignancies. This limits statistical inference but the population is still informative for directional signal assessment.

In [5]:
# Manually curated list of confirmed appendiceal cancer indication terms
# Non-cancer and non-appendiceal matches from Step 1 have been excluded (see markdown above)
cancer_terms = [
    'Adenocarcinoma of appendix',
    'Appendix cancer',
    'Appendix cancer metastatic',
    'Mucinous adenocarcinoma of appendix',
    'Pseudomyxoma peritonei',
]

# Build appendiceal_reports table to store just the unique primaryids for this population
# All subsequent notebooks (reactions, population, outcomes) join against this table
# Same pattern as fluorouracil_analysis in Module 1
placeholders = ','.join(['?' for _ in cancer_terms])

conn.execute("DROP TABLE IF EXISTS appendiceal_reports;")
conn.execute(f"""
    CREATE TABLE appendiceal_reports AS
    SELECT DISTINCT primaryid
    FROM indi
    WHERE indi_pt IN ({placeholders})
""", cancer_terms)
conn.commit()

# Confirm row count
total = conn.execute("SELECT COUNT(*) FROM appendiceal_reports").fetchone()[0]
print(f"Appendiceal reports created: {total:,}")

Appendiceal reports created: 104


## Step 3: Drug Landscape

What drugs are most commonly reported in appendiceal cancer patients? This gives us a picture of the treatment landscape and tells us which drugs are worth profiling in the reactions and outcomes notebooks.

In [6]:
# Top drugs reported in appendiceal cancer patients
# Join appendiceal_reports (our filtered primaryid set) to the drug table
# role_cod PS/SS = primary or secondary suspect drug — excludes concomitant medications
top_drugs = pd.read_sql_query("""
    SELECT d.drugname, COUNT(DISTINCT d.primaryid) AS reports
    FROM drug d
    JOIN appendiceal_reports a ON d.primaryid = a.primaryid
    WHERE d.role_cod IN ('PS', 'SS')
    GROUP BY d.drugname
    ORDER BY reports DESC
    LIMIT 25
""", conn)

top_drugs

,drugname,reports
0,CAPECITABINE,32
1,OXALIPLATIN,31
2,FLUOROURACIL,24
3,BEVACIZUMAB,20
4,LEUCOVORIN,16
5,FRUQUINTINIB,12
6,IRINOTECAN HYDROCHLORIDE,9
7,LONSURF,8
8,ZEJULA,7
9,OPDIVO,4


## Step 3 Observations: Drug Landscape in Appendiceal Cancer Reports

The top reported drugs reflect the standard treatment landscape for appendiceal cancer and colorectal malignancies:

| Drug | Reports | Class | Notes |
|---|---|---|---|
| Capecitabine | 32 | Fluoropyrimidine (oral) | Oral prodrug of fluorouracil; same mechanism, used when IV 5-FU is impractical |
| Oxaliplatin | 31 | Platinum-based chemo | Core component of FOLFOX and HIPEC regimens |
| Fluorouracil | 24 | Fluoropyrimidine (IV) | Classic 5-FU; analyzed in full in Module 1 |
| Bevacizumab | 20 | VEGF inhibitor (biologic) | Anti-angiogenic; added to FOLFOX/FOLFIRI in metastatic colorectal cancer |
| Leucovorin | 16 | Chemo modulator | Not chemotherapy itself. Potentiates 5-FU activity; always used alongside it |
| Fruquintinib | 12 | VEGFR inhibitor (targeted) | Newer targeted agent approved for refractory colorectal cancer |
| Irinotecan / Irinotecan HCl | 13 | Topoisomerase inhibitor | Core component of FOLFIRI; same drug, two name variants |
| Lonsurf | 8 | Fluoropyrimidine combo (oral) | Brand name for trifluridine/tipiracil; used in refractory colorectal cancer |
| Zejula | 7 | PARP inhibitor | Niraparib; primarily ovarian cancer. Likely reflecting peritoneal disease overlap |
| Opdivo | 4 | PD-1 checkpoint inhibitor | Nivolumab; immunotherapy used in MSI-high colorectal/appendiceal tumors |
| Losartan | 4 | Antihypertensive | Off-target concomitant medication. Not an oncology drug |
| Avastin | 3 | VEGF inhibitor (biologic) | Brand name for bevacizumab; same drug as row 4 |
| Yervoy | 2 | CTLA-4 checkpoint inhibitor | Ipilimumab; immunotherapy, often combined with Opdivo |
| Vectibix | 2 | EGFR inhibitor (biologic) | Panitumumab; used in RAS wild-type colorectal cancer |
| Norepinephrine | 2 | Vasopressor | ICU medication — likely reported during post-surgical or septic events |
| Mitomycin C / Mitomycin | 4 | Alkylating agent | Key HIPEC drug; low report count likely reflects intraperitoneal route underreporting |
| Herceptin | 2 | HER2 inhibitor (biologic) | Trastuzumab; HER2-targeted therapy, occasionally relevant in colorectal/gastric |
| Braftovi | 2 | BRAF inhibitor (targeted) | Encorafenib; used in BRAF V600E-mutant colorectal cancer |
| Tislelizumab | 1 | PD-1 checkpoint inhibitor | Investigational immunotherapy |
| Stivarga | 1 | Multi-kinase inhibitor | Regorafenib; refractory colorectal cancer |
| Palbociclib | 1 | CDK4/6 inhibitor | Primarily breast cancer — likely off-label or concomitant |
| Octreotide acetate | 1 | Somatostatin analogue | Used in carcinoid/neuroendocrine tumors — relevant given carcinoid overlap in this population |
| Leucovorin calcium | 1 | Chemo modulator | Same drug as Leucovorin above — name variant |

> **Notable:** Mitomycin C has only 4 reports despite being a primary HIPEC drug. This likely reflects underreporting of intraperitoneal chemotherapy in FAERS, since HIPEC is administered surgically rather than through standard outpatient channels where adverse event reporting is more routine.


## Step 4: Build Consolidated Appendiceal Cohort Table

We extend `appendiceal_reports` (currently just primaryids) into a richer analysis table
by joining in drug and reaction information. Both tables join on `primaryid` and return
one row per drug-reaction combination per report.

We filter `drug` to role_cod IN ('PS', 'SS') to keep only primary and secondary suspect
drugs — same filter applied in Module 1.

The resulting `appendiceal_cohort` table is the base table used across notebooks 02–04.


In [11]:
# Build appendiceal_cohort table by joining indi, drug, and reac on primaryid
# indi provides the indication filter and indi_pt label
# drug provides drugname and role_cod (filtered to PS/SS suspect drugs only)
# reac provides the reaction term (pt)
# Result: one row per drug-reaction combination per report

conn.execute("DROP TABLE IF EXISTS appendiceal_cohort;")
conn.execute(f"""
    CREATE TABLE appendiceal_cohort AS
    SELECT
        i.primaryid,
        i.indi_pt,
        d.drugname,
        d.role_cod,
        r.pt AS reaction
    FROM (SELECT DISTINCT primaryid, indi_pt FROM indi) i
    JOIN (SELECT DISTINCT primaryid, drugname, role_cod FROM drug) d 
        ON i.primaryid = d.primaryid
    JOIN (SELECT DISTINCT primaryid, pt FROM reac) r 
        ON i.primaryid = r.primaryid
    WHERE i.indi_pt IN ({placeholders})
      AND d.role_cod IN ('PS', 'SS')
""", cancer_terms)
conn.commit()

total = conn.execute("SELECT COUNT(*) FROM appendiceal_cohort").fetchone()[0]
print(f"Appendiceal cohort rows: {total:,}")





Appendiceal cohort rows: 879
